# Experiment report template (KAN-81)

Copy to `notebooks/<ticket>-<name>.ipynb`. The eight sections follow protocol section 13. Numbers come from code or MLflow, never typed by hand.
Render and attach to the run: `uv run --group experiments python -m pm25_forecast.experiments.tracking render-report notebooks/<file>.ipynb --run-id <RUN_ID> --execute`
Strip outputs before commit: `uv run --group experiments jupyter nbconvert --clear-output --inplace notebooks/<file>.ipynb`

In [ ]:
from pathlib import Path

from pm25_forecast.experiments import tracking
from pm25_forecast.experiments.prereg import load_prereg

# Parameters: edit only this cell.
TICKET = "KAN-0"
AGENT = "claude-data-scientist"
EXPERIMENT = "kan78/e00-template"
RUN_NAME = "template-run"
PREREG_ID = "e00-template"
SNAPSHOT_DIR = "~/pm25-experiments/snapshots/e02-v1"
SPLIT = "dev"  # dev | holdout

## 1. Summary

One paragraph: question, answer, decision. Numbers come from the table in section 5.

## 2. Hypothesis and decision rule

Copy from the committed prereg file (`docs/experiments/prereg/<id>.yaml`). Do not edit after the run.

## 3. Setup

Snapshot hash, folds, refit schedule, features (protocol sections 3, 4).

In [ ]:
prereg_path = next(Path("docs/experiments/prereg").glob(f"{PREREG_ID}.*"), None)
prereg = load_prereg(prereg_path) if prereg_path else None
snapshot_sha256 = tracking.snapshot_hash(Path(SNAPSHOT_DIR).expanduser())
print(tracking.tracking_uri(), snapshot_sha256)

## 4. EDA and data checks

Coverage, era exclusions, serving-lag check.

## 5. Results table

MLflow run ids and metrics (cell below reads them from the store).

In [ ]:
# Fit and score with the shared feature module; fill `metrics` from results, then log once.
metrics = {"mae_24h": float("nan"), "rmse_24h": float("nan")}  # placeholders: log_run refuses them
run_id = tracking.log_run(
    EXPERIMENT,
    RUN_NAME,
    params={"seed": 0},
    metrics=metrics,
    tags={
        "ticket": TICKET,
        "agent": AGENT,
        "model_name": prereg.model if prereg else "unset",
        "model_family": prereg.model_family if prereg else "unset",
        "feature_set": prereg.feature_set if prereg else "unset",
        "snapshot_sha256": snapshot_sha256,
        "split": SPLIT,
        "prereg": PREREG_ID,
    },
)
print(run_id)

## 6. Statistical tests and diagnostics

Paired CI, Diebold-Mariano, sensitivities, station gate (protocol sections 6, 10).

## 7. Figures

Every figure is produced by code in this notebook.

## 8. Conclusion, limitations, next

Against the decision rule; what would change the conclusion.